# Партиционирование в Iceberg: hidden partitioning и partition evolution

В отличие от классического Hive-партиционирования (где партиция — это часть
пути `.../year=2024/month=01/...` и пользователь обязан сам подставлять эти
значения в фильтры), Iceberg использует **скрытое партиционирование (hidden
partitioning)**:

- партиция вычисляется автоматически из значения колонки через *transform*
  (`identity`, `bucket(N, col)`, `truncate(N, col)`, `year(col)`,
  `month(col)`, `day(col)`, `hour(col)`);
- пользователю никогда не нужно писать `WHERE year(event_time) = 2024` — Iceberg
  сам умеет "срезать" (`partition pruning`) ненужные файлы по обычному
  предикату `WHERE event_time >= ...`;
- партиционирование можно **менять со временем** (partition evolution) без
  переписывания старых данных: старые файлы остаются со старой партиционной
  схемой, новые пишутся по новой.

In [1]:
from pyspark.sql import SparkSession

# Настраиваем каталог Iceberg прямо в коде, а не через spark-defaults.conf.
# Это даёт тот же результат, но позволяет менять параметры каталога
# независимо от конфигурации образа и делает ноутбук самодостаточным.
spark = (
    SparkSession.builder
    .appName("Partitioning")
    .master("local[*]")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.course", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.course.type", "rest")
    .config("spark.sql.catalog.course.uri", "http://rest-catalog:8181")
    .config("spark.sql.catalog.course.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.course.warehouse", "s3://warehouse")
    .config("spark.sql.catalog.course.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.course.s3.path-style-access", "true")
    .getOrCreate()
)

print("Spark + Iceberg готовы к работе")

Spark + Iceberg готовы к работе


Ниже определена вспомогательная функция `read_metadata_table` для чтения
служебных metadata-таблиц Iceberg в обход REST-каталога этого стенда —
подробное объяснение, зачем она нужна, см. в `00_Setup_and_Basics.ipynb`.

In [2]:
from pyspark.sql import DataFrame


def read_metadata_table(spark, table_identifier, metadata_type):
    """Читает metadata-таблицу Iceberg (history, snapshots, files, partitions, manifests, ...)
    в обход REST-каталога, который отклоняет составные namespace в URL
    (см. подробное объяснение в 00_Setup_and_Basics.ipynb)."""
    jvm = spark._jvm
    table = jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(spark._jsparkSession, table_identifier)
    mtype = jvm.org.apache.iceberg.MetadataTableType.valueOf(metadata_type.upper())
    jdf = jvm.org.apache.iceberg.spark.SparkTableUtil.loadMetadataTable(spark._jsparkSession, table, mtype)
    return DataFrame(jdf, spark)

In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS course.test")

spark.sql("DROP TABLE IF EXISTS course.test.events")
spark.sql("""
CREATE TABLE course.test.events (
    event_id    BIGINT,
    user_id     BIGINT,
    event_type  STRING,
    event_time  TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(event_time), bucket(4, user_id))
""")

DataFrame[]

Таблица партиционирована по дню события (`days(event_time)`) и по 4
бакетам от `user_id` — это распространённая комбинация: "срез по времени" +
"равномерное распределение по ключу", чтобы избежать перекоса (data skew) при
записи и при join.

In [4]:
from datetime import datetime

rows = [
    (1, 100, "click", datetime(2024, 1, 1, 10, 0)),
    (2, 101, "click", datetime(2024, 1, 1, 12, 30)),
    (3, 100, "view",  datetime(2024, 1, 2, 9, 15)),
    (4, 102, "view",  datetime(2024, 1, 2, 18, 45)),
    (5, 103, "click", datetime(2024, 1, 3, 8, 0)),
]
df = spark.createDataFrame(rows, ["event_id", "user_id", "event_type", "event_time"])
df.writeTo("course.test.events").append()

spark.sql("SELECT * FROM course.test.events ORDER BY event_id").show(truncate=False)

+--------+-------+----------+-------------------+
|event_id|user_id|event_type|event_time         |
+--------+-------+----------+-------------------+
|1       |100    |click     |2024-01-01 10:00:00|
|2       |101    |click     |2024-01-01 12:30:00|
|3       |100    |view      |2024-01-02 09:15:00|
|4       |102    |view      |2024-01-02 18:45:00|
|5       |103    |click     |2024-01-03 08:00:00|
+--------+-------+----------+-------------------+



## Смотрим физические партиции через metadata-таблицу `.partitions`

Значения партиций (номер дня, номер бакета) вычислены Iceberg автоматически —
их не нужно было указывать при записи.

In [5]:
read_metadata_table(spark, "course.test.events", "partitions") \
    .select("partition", "record_count", "file_count") \
    .show(truncate=False)

+---------------+------------+----------+
|partition      |record_count|file_count|
+---------------+------------+----------+
|{2024-01-03, 3}|1           |1         |
|{2024-01-02, 0}|2           |1         |
|{2024-01-01, 0}|2           |1         |
+---------------+------------+----------+



## Partition pruning: обычный фильтр по колонке уже "просто работает"

Не нужно ничего знать про transform — Spark сам сопоставит `event_time >=
...` со значениями `days(event_time)` в манифестах и отбросит ненужные
файлы ещё на этапе планирования.

In [6]:
spark.sql("""
SELECT * FROM course.test.events
WHERE event_time >= TIMESTAMP '2024-01-02 00:00:00'
""").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- BatchScan course.test.events[event_id#101L, user_id#102L, event_type#103, event_time#104] course.test.events (branch=null) [filters=event_time IS NOT NULL, event_time >= 1704153600000000, groupedBy=] RuntimeFilters: []




## Partition evolution

Допустим, объём данных вырос и партиционирование по дню стало слишком
крупным — переходим на партиционирование по часу, а бакетирование по
`user_id` меняем с 4 на 8 бакетов. Старые данные **не переписываются** — они
остаются в старых партициях, а вот все новые записи лягут уже по новой схеме.

In [7]:
spark.sql("ALTER TABLE course.test.events DROP PARTITION FIELD days(event_time)")
spark.sql("ALTER TABLE course.test.events ADD PARTITION FIELD hours(event_time)")

spark.sql("ALTER TABLE course.test.events DROP PARTITION FIELD bucket(4, user_id)")
spark.sql("ALTER TABLE course.test.events ADD PARTITION FIELD bucket(8, user_id)")

DataFrame[]

In [11]:
more_rows = [
    (6, 104, "click", datetime(2024, 1, 4, 8, 5)),
    (7, 105, "view",  datetime(2024, 1, 4, 8, 40)),
]
df2 = spark.createDataFrame(more_rows, ["event_id", "user_id", "event_type", "event_time"])
df2.writeTo("course.test.events").append()

# В partitions теперь видны и старые (по дню), и новые (по часу) партиции —
# это и есть суть partition evolution: смешанные partition spec'и уживаются в одной таблице.
read_metadata_table(spark, "course.test.events", "partitions") \
    .select("partition", "spec_id", "record_count") \
    .show(truncate=False)

AttributeError: 'NoneType' object has no attribute 'sc'

## Список partition spec-ов таблицы

Каждое изменение партиционирования создаёт новый `partition spec` с
собственным `spec_id`. Старые data-файлы помнят, по какому spec-у они были
записаны — это и позволяет не переписывать историю.

In [9]:
read_metadata_table(spark, "course.test.events", "files") \
    .select("spec_id").distinct() \
    .show()

+-------+
|spec_id|
+-------+
|      4|
|      0|
+-------+



In [12]:
spark.stop()